# Colab: development of a gender classification model 

## Set-up

In [ ]:
!pip install -U datasets
!pip install ray

In [ ]:
import datasets
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import random
import numpy as np
from datasets import Dataset

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Clear PyTorch GPU cache
torch.cuda.empty_cache()

# Training set parameters
fraction_positives = 1
fraction_positives_in_test = 0.1        # percentage of positives in test set
fraction_remaining_unlabeled = 0.2      # percentage to keep of unlabled for reconstructing training set

# Model parameters
model_id = "xlm-roberta-large"
max_token_length = None                 # None uses max model allows

# Hyperparameter tuning
fraction_positives_for_tuning = 1
n_trials_ray = 20
n_cpu_ray = 2
n_gpu_ray = 1                           # For colab, 1 GPU max

# Prediction parameters
fraction_unlabeled_for_prediction = 0.1

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Initial classification

In [ ]:
# Load the data
gen_to_mine = pd.read_feather("./drive/MyDrive/Colab Notebooks/gen_to_mine.feather")

# Discard any missing values in text_mining_description for robustness
gen_to_mine = gen_to_mine.dropna(subset=['text_mining_description'])

### Construct training set with random negatives

In [ ]:
# Filter out positive and unlabeled samples
# Drop single-keyword-only rows like ['woman'] to avoid trivial positives
positive = gen_to_mine[gen_to_mine['is_gender'] == True].copy()
unlabeled = gen_to_mine[gen_to_mine['is_gender'] == False].copy()

# Sample randomly fraction_positives for positives
positive = positive.sample(frac=fraction_positives, random_state=SEED)

# Stratified sampling from unlabeled
positive.loc[:, 'length_bin'] = pd.cut(positive['text_mining_description'].str.len(), bins=6, labels=False, include_lowest=True)
unlabeled.loc[:, 'length_bin'] = pd.cut(unlabeled['text_mining_description'].str.len(), bins=6, labels=False, include_lowest=True)

positive.loc[:, 'stratum'] = positive['language'].astype(str) + "_" + positive['length_bin'].astype(str)
unlabeled.loc[:, 'stratum'] = unlabeled['language'].astype(str) + "_" + unlabeled['length_bin'].astype(str)

stratum_counts = positive['stratum'].value_counts(normalize=True)

samples_per_stratum = (stratum_counts * len(positive)).round().astype(int)

neg_samples = []
for stratum, n in samples_per_stratum.items():
    candidates = unlabeled[unlabeled['stratum'] == stratum]
    if len(candidates) >= n:
        neg_samples.append(candidates.sample(n=n, random_state=SEED))
    else:
        neg_samples.append(candidates)

unlabeled_sampled = pd.concat(neg_samples).sample(frac=1, random_state=SEED)

# Combine positives and sampled negatives
balanced = pd.concat([positive, unlabeled_sampled]).reset_index(drop=True)

# Keep the remaining unlabeled samples for later
remaining_unlabeled = unlabeled.drop(unlabeled_sampled.index)

In [ ]:
# Split into train & test sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    balanced['text_mining_description'],
    balanced['is_gender'].astype(int),
    test_size=0.1,
    stratify=balanced['is_gender'],
    random_state=SEED,
)

balanced['is_gender'].value_counts()

### Tokenization

In [ ]:
# Load tokenizer and tokenize data
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=max_token_length)

# Create Hugging Face datasets
train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'label': train_labels.tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'label': test_labels.tolist()})

# Tokenize dataset with custom tokenize()
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

### Model training

In [ ]:
# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./drive/MyDrive/Colab Notebooks/models/gender/results",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=1e-5,
    logging_dir="./drive/MyDrive/Colab Notebooks/models/gender/logs",
    eval_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    save_total_limit=1, # keeps only the latest checkpoint
    metric_for_best_model="precision",
    fp16=True,
    report_to="none",
    seed=SEED
)

# Define metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

# Predict on test
predictions_output = trainer.predict(test_dataset)

# Output test set with text, label, predicted probabilities
test_results = pd.DataFrame({
    'text': test_texts,
    'label': test_labels,
    'probability_is_gender': torch.softmax(torch.tensor(predictions_output.predictions), dim=1)[:, 1].numpy()
})

### Model evaluation

In [ ]:
# ROC and PR Curves
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score
import plotly.graph_objs as go
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

# Extract labels and predicted probabilities
y_true = test_results['label'].values
y_scores = test_results['probability_is_gender'].values

# ROC
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = roc_auc_score(y_true, y_scores)
hover_text = [f"Threshold: {thr:.3f}<br>FPR: {f:.3f}<br>TPR: {t:.3f}" for thr, f, t in zip(thresholds, fpr, tpr)]
trace = go.Scatter(x=fpr, y=tpr, mode='lines+markers', text=hover_text, hoverinfo='text', name=f'ROC curve (AUC = {roc_auc:.3f})', line=dict(color='orange'))
line_random = go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Guessing', line=dict(dash='dash', color='navy'))
layout = go.Layout(title='ROC Curve', xaxis=dict(title='False Positive Rate'), yaxis=dict(title='True Positive Rate (Recall)'), legend=dict(x=0.6, y=0.05), hovermode='closest')
fig = go.Figure(data=[trace, line_random], layout=layout)
fig.show()

In [ ]:
# PR
precision, recall, pr_thresholds = precision_recall_curve(y_true, y_scores)
pr_auc = average_precision_score(y_true, y_scores)
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'PR curve (AP = {pr_auc:.3f})', color='green', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall Curve')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

## Rebuild training set with reliable negatives

### Predict unlabeled

Predict the part of the unlabeled set to obtain reliable negatives for reconstructing the training set later on

In [ ]:
# Sample from remaining unlabeled
remaining_unlabeled = remaining_unlabeled.sample(frac=fraction_remaining_unlabeled, random_state=SEED)

# Construct dataset
remaining_unlabeled_texts = remaining_unlabeled["text_mining_description"].tolist()
remaining_unlabeled_dataset = Dataset.from_dict({'text': remaining_unlabeled_texts})
remaining_unlabeled_dataset = remaining_unlabeled_dataset.map(tokenize, batched=True)

# Predict the unlabeled samples
pred_output = trainer.predict(remaining_unlabeled_dataset)
logits = pred_output.predictions
probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
remaining_unlabeled["probability_is_gender"] = probs

# Save to xlsx to inspect manually
remaining_unlabeled.to_excel("./drive/MyDrive/Colab Notebooks/unlabeled_predictions_gender.xlsx", index=False)

### Rebuild training set with reliable negatives

In [ ]:
# --------------------------- Positives ----------------------------------------
positive = gen_to_mine[gen_to_mine['is_gender'] == True].copy()
positive = positive.sample(frac=fraction_positives_for_tuning, random_state=SEED)

# Use the same stratum distribution as positives
positive.loc[:, 'length_bin'] = pd.cut(
    positive['text_mining_description'].str.len(), bins=6, labels=False, include_lowest=True
)
positive.loc[:, 'stratum'] = positive['language'].astype(str) + "_" + positive['length_bin'].astype(str)
stratum_counts = positive['stratum'].value_counts(normalize=True)

# ------------------------- Reliable negatives ---------------------------------
# Select reliable negatives: probability < 0.8
reliable_negatives = remaining_unlabeled[remaining_unlabeled['probability_is_gender'] < 0.8].copy()

samples_per_stratum = (stratum_counts * len(positive)).round().astype(int)
neg_samples = []
for stratum, n in samples_per_stratum.items():
    candidates = reliable_negatives[reliable_negatives['stratum'] == stratum]
    if len(candidates) >= n:
        neg_samples.append(candidates.sample(n=n, random_state=SEED))
    else:
        neg_samples.append(candidates)

reliable_negatives_sampled = pd.concat(neg_samples).sample(frac=1, random_state=SEED)
reliable_negatives_sampled = reliable_negatives_sampled.drop(columns=['probability_is_gender', 'length_bin', 'stratum'])

# Rebuild the training set
positive = positive.drop(columns=['length_bin', 'stratum'])
training_dataset_with_reliables = pd.concat([positive, reliable_negatives_sampled]).reset_index(drop=True)
training_dataset_with_reliables = training_dataset_with_reliables.drop(columns=['is_mining'], errors='ignore')

training_dataset_with_reliables['is_gender'].value_counts()

# Split into train+val and test sets first (90/10)
train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    training_dataset_with_reliables['text_mining_description'],
    training_dataset_with_reliables['is_gender'].astype(int),
    test_size=0.1,
    stratify=training_dataset_with_reliables['is_gender'],
    random_state=SEED,
)

# Now split train+val into train and validation sets (80/10/10 overall)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts,
    train_val_labels,
    test_size=1/9,
    stratify=train_val_labels,
    random_state=SEED,
)

# Optionally augment val/test to adjust prevalence (mirroring stats)
n_pos_val = sum(val_labels)
n_neg_val = len(val_labels) - n_pos_val
n_neg_add_hyper_tuning = int((n_pos_val - fraction_positives_in_test*(n_pos_val + n_neg_val))/fraction_positives_in_test)

additional_negatives_val = remaining_unlabeled.sample(n=max(n_neg_add_hyper_tuning,0), random_state=SEED)
additional_negatives_test = remaining_unlabeled.sample(n=max(n_neg_add_hyper_tuning,0), random_state=SEED)

val_texts = pd.concat([val_texts, additional_negatives_val['text_mining_description']])
val_labels =  pd.concat([val_labels, additional_negatives_val['is_gender'].astype(int)])

test_texts = pd.concat([test_texts, additional_negatives_test['text_mining_description']])
test_labels =  pd.concat([test_labels, additional_negatives_test['is_gender'].astype(int)])

print(f"Final–val/test hyper tuning prevalence: {np.mean(val_labels)}/{np.mean(test_labels)}")
print(f"Number of positives in val/test set: {sum(val_labels)}/{sum(test_labels)}")

In [ ]:
# Tokenizer for train/val/test
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=max_token_length)

train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'label': train_labels.tolist()})
val_dataset = Dataset.from_dict({'text': val_texts.tolist(), 'label': val_labels.tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'label': test_labels.tolist()})

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

## Hyperparameter tuning

In [ ]:
import os
import ray
from ray import tune
from ray.tune import JupyterNotebookReporter
from ray.tune.schedulers import ASHAScheduler
from transformers import TrainerCallback

reporter = JupyterNotebookReporter(
    parameter_columns=["learning_rate", "per_device_train_batch_size"],
    metric_columns=["eval_accuracy", "eval_f1", "eval_auc", "eval_loss", "epoch"],
)

scheduler = ASHAScheduler(
        metric="accuracy",
        mode="max",
        grace_period=1,
        reduction_factor=10)

# Define metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    preds = np.argmax(logits, axis=-1)

    # Get probability for positive class (class 1)
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

    # Standard classification metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)

    # Compute AUC
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')

    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall, "auc": auc}

class TuneReportCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            tune.report(metrics)

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

config = {
    "learning_rate": tune.uniform(1e-5, 5e-5),
    "weight_decay":  tune.uniform(0.0, 0.3),
    "num_train_epochs": tune.choice([2, 3, 4]),
    "per_device_train_batch_size": tune.choice([8, 16, 32]),
    "warmup_ratio": tune.choice([0.0, 0.06, 0.1]),
    "gradient_accumulation_steps": tune.choice([1, 2, 4]),
    "adam_beta1": tune.choice([0.9, 0.95]),
    "adam_beta2": tune.choice([0.98, 0.999]),
}

train_ref = ray.put(train_dataset)
val_ref = ray.put(val_dataset)

def trainer_init(config):
    output_dir = os.path.abspath("./models/results")

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["per_device_train_batch_size"],
        gradient_accumulation_steps = config["gradient_accumulation_steps"],
        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],
        adam_beta1=config["adam_beta1"],
        adam_beta2=config["adam_beta2"],
        per_device_eval_batch_size=8,
        logging_dir="./models/logs",
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        metric_for_best_model="auc",
        report_to="none",
        seed=SEED,
    )

    return Trainer(
        args=args,
        train_dataset=ray.get(train_ref),
        eval_dataset=ray.get(val_ref),
        compute_metrics=compute_metrics,
        model_init=model_init,
        callbacks=[TuneReportCallback()]
    )


def run_ray_train(config):
    trainer = trainer_init(config)
    trainer.train()
    eval_metrics = trainer.evaluate()
    tune.report(eval_metrics)

storage_path = os.path.abspath("./drive/MyDrive/Colab Notebooks/ray_tune_results/gender")

# Hyperparameter tuning with Ray Tune
analysis = tune.run(
    run_ray_train,
    config=config,
    resources_per_trial={"cpu": n_cpu_ray, "gpu": n_gpu_ray},
    metric="eval_accuracy",
    mode="max",
    num_samples=n_trials_ray,
    # scheduler=scheduler,
    progress_reporter=reporter,
    storage_path=storage_path,
    name="tune_xlm_roberta_gender"
)

In [ ]:
# Best configs manually added after inspecting the ray tune results
best_config_base = {
  "adam_beta1": 0.95,
  "adam_beta2": 0.98,
  "gradient_accumulation_steps": 2,
  "learning_rate": 3.5175945525410505e-05,
  "num_train_epochs": 4,
  "per_device_train_batch_size": 8,
  "warmup_ratio": 0.0,
  "weight_decay": 0.20872460669538515,
}

best_config_large = {
  "adam_beta1": 0.9,
  "adam_beta2": 0.999,
  "gradient_accumulation_steps": 4,
  "learning_rate": 2.5023318105597762e-05,
  "num_train_epochs": 4,
  "per_device_train_batch_size": 32,
  "warmup_ratio": 0.06,
  "weight_decay": 0.028194581952260697,
}

## Final classification of full unlabeled set

### Retraining with best config from hyperparameter search

In [ ]:
# Final training split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    training_dataset_with_reliables['text_mining_description'],
    training_dataset_with_reliables['is_gender'].astype(int),
    test_size=0.1,
    stratify=training_dataset_with_reliables['is_gender'],
    random_state=SEED,
)

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=max_token_length)

train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'label': train_labels.tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'label': test_labels.tolist()})

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Model and args
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

best_config = best_config_large.copy()

training_args = TrainingArguments(
    output_dir="./drive/MyDrive/Colab Notebooks/models/gender/results",
    num_train_epochs=best_config["num_train_epochs"],
    per_device_train_batch_size=best_config["per_device_train_batch_size"],
    per_device_eval_batch_size=64,
    gradient_accumulation_steps = best_config["gradient_accumulation_steps"],
    warmup_ratio=best_config["warmup_ratio"],
    weight_decay=best_config["weight_decay"],
    learning_rate=best_config["learning_rate"],
    adam_beta1=best_config["adam_beta1"],
    adam_beta2=best_config["adam_beta2"],
    logging_dir="./drive/MyDrive/Colab Notebooks/models/gender/logs",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1,
    metric_for_best_model="auc",
    fp16=True,
    report_to="none",
    seed=SEED,
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall, "auc": auc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

predictions_output = trainer.predict(test_dataset)

test_results = pd.DataFrame({
    'text': test_texts,
    'label': test_labels,
    'probability_is_gender': torch.softmax(torch.tensor(predictions_output.predictions), dim=1)[:, 1].numpy()
})

### Prediction of full unlabeled set

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=max_token_length)

# Load unlabeled set
unlabeled = gen_to_mine[gen_to_mine['is_gender'] == False].copy()
unlabeled_texts = unlabeled["text_mining_description"].tolist()
unlabeled_dataset = Dataset.from_dict({'text': unlabeled_texts})
unlabeled_dataset = unlabeled_dataset.map(tokenize, batched=True)

# Load conflicting descriptions for gender
conflicting_gen = pd.read_feather("./drive/MyDrive/Colab Notebooks/conflicting_descr_gen.feather")
conflicting_gen_texts = conflicting_gen["text_mining_description"].tolist()
conflicting_gen_dataset = Dataset.from_dict({'text': conflicting_gen_texts})
conflicting_gen_dataset = conflicting_gen_dataset.map(tokenize, batched=True)

If trainer no longer in memory, load from model checkpoint created during section "Retraining with best config from hyperparameter search". If it is still loaded, directly predict the unlabled set with the cell beneath.

In [ ]:
# If trainer not in memory, load best checkpoint manually
# model_checkpoint = "./drive/MyDrive/Colab Notebooks/models/gender/results/large-checkpoint-XXXX"  # adjust
# model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint)
args = TrainingArguments(
    output_dir="./results",             # dummy
    per_device_eval_batch_size=128,
    do_train=False,
    do_eval=False,
    fp16=True,
    report_to="none",
    logging_strategy="no",
    seed=SEED,
)

inference_trainer = Trainer(
    model=model,
    args=args
)

In [ ]:
# Predict unlabeled
pred_output = inference_trainer.predict(unlabeled_dataset)
logits = pred_output.predictions
probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
unlabeled['probability_is_gender'] = probs

In [ ]:
unlabeled.to_feather("./drive/MyDrive/Colab Notebooks/unlabeled_predicted_gen.feather")

In [ ]:
# Predict conflicting set 
pred_output_conflicting = inference_trainer.predict(conflicting_gen_dataset)
logits_conflicting = pred_output_conflicting.predictions
probs_conflicting = torch.softmax(torch.tensor(logits_conflicting), dim=1)[:, 1].numpy()
conflicting_gen['probability_is_gender'] = probs_conflicting
conflicting_gen.to_feather("./drive/MyDrive/Colab Notebooks/conflicting_predicted_gen.feather")